# Exercise File 3 — Hard & Advanced Concepts (Fresh Variation)
**Concepts:** Conversion Rate by Segment · Date Overlap (double-booking) · Avg Resolution Time · Merge Monthly Reports · Multi-Agg Lambda · Server Log Parsing · Custom Sort · Top-K heapq · Lag/Lead · Rank

---
## Q1 — Top 3 Income Brackets by Ad Conversion Rate (Hard)

**Concept:** merge · age//10 style bucketing · conditional groupby agg · sort with tiebreak

**Problem:** You have `users` (with annual income) and `ad_events` (impression or conversion). Find the **top 3 income brackets** (bucketed by $20k bands) with the highest conversion rate. Conversion rate = conversions / total impressions. Tiebreak: higher income bracket ranks first.

**Sample Input — users:**

| user_id | income |
|---------|--------|
| 1       | 25000  |
| 2       | 45000  |
| 3       | 25000  |
| 4       | 65000  |

**Sample Input — ad_events:**

| event_id | user_id | year | action      |
|----------|---------|------|-------------|
| 1        | 1       | 2023 | impression  |
| 2        | 1       | 2023 | conversion  |
| ...      | ...     | ...  | ...         |

**Sample Output:**

| income_bracket | conversions | total | conv_rate |
|----------------|-------------|-------|-----------|
| 3 (60k-80k)    | ...         | ...   | highest   |
| 1 (20k-40k)    | ...         | ...   |           |
| 2 (40k-60k)    | ...         | ...   |           |

In [ ]:
import pandas as pd

users = pd.DataFrame({
    'user_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'income':  [25000, 45000, 25000, 65000, 45000, 65000, 25000, 15000]
})
ad_events = pd.DataFrame({
    'event_id': range(1, 21),
    'user_id':  [1,1,1,1, 2,2,2, 3,3, 4,4,4,4, 5,5, 6, 7,7,7, 8],
    'year':     [2023]*20,
    'action':   [
        'impression','conversion','impression','impression',  # U1 (25k): 1c/3
        'impression','impression','conversion',               # U2 (45k): 1c/2
        'impression','conversion',                           # U3 (25k): 1c/1
        'impression','conversion','impression','conversion', # U4 (65k): 2c/2
        'impression','conversion',                           # U5 (45k): 1c/1
        'conversion',                                        # U6 (65k): 1c/1
        'impression','impression','impression',              # U7 (25k): 0c/3
        'conversion',                                        # U8 (15k): 1c/1
    ]
})
print(users)


**Concepts to use:**
1. `merge()` — join events with users.
2. `income // 20000` — bucket into $20k brackets.
3. Lambda agg: `(x=='conversion').sum()` for conversions within groupby.
4. `sort_values(['conv_rate','bracket'], ascending=[False,False]).head(3)` — tiebreak.

In [ ]:
# Optimised Solution
def top_brackets_by_conversion(events, users):
    df = events.merge(users, on='user_id')
    df = df[df['year'] == 2023].copy()
    df['income_bracket'] = df['income'] // 20000
    agg = df.groupby('income_bracket')['action'].agg(
        conversions=lambda x: (x == 'conversion').sum(),
        total=lambda x: x.count()
    ).reset_index()
    agg['conv_rate'] = agg['conversions'] / agg['total']
    return agg.sort_values(['conv_rate','income_bracket'], ascending=[False,False]).head(3)

print(top_brackets_by_conversion(ad_events, users))


---
## Q2 — Detect Hotel Room Double-Bookings (Hard)

**Concept:** self-join on room_id · pair filter (id_a < id_b) · overlap condition

**Problem:** A hotel has a booking table. Find all `room_id`s that have at least one **pair of overlapping bookings** (i.e., a double-booking). Two bookings overlap if `checkin_a < checkout_b AND checkin_b < checkout_a`.

**Sample Input:**

| booking_id | room_id | checkin    | checkout   |
|------------|---------|------------|------------|
| B1         | R1      | 2024-03-01 | 2024-03-05 |
| B2         | R1      | 2024-03-04 | 2024-03-08 |
| B3         | R2      | 2024-03-01 | 2024-03-03 |
| B4         | R2      | 2024-03-03 | 2024-03-07 |
| B5         | R3      | 2024-03-01 | 2024-03-10 |
| B6         | R3      | 2024-03-09 | 2024-03-12 |

**Sample Output:**

| room_id | has_double_booking |
|---------|--------------------|
| R1      | True               |
| R2      | False              |
| R3      | True               |

> R1: B1 checkout Mar 5, B2 checkin Mar 4 → overlap ✅ · R2: B3 checkout = B4 checkin (back-to-back) → no overlap ❌ · R3: B5 checkout Mar 10, B6 checkin Mar 9 → overlap ✅

In [ ]:
import pandas as pd

bookings = pd.DataFrame({
    'booking_id': ['B1','B2','B3','B4','B5','B6','B7'],
    'room_id':    ['R1','R1','R2','R2','R3','R3','R4'],
    'checkin':    pd.to_datetime(['2024-03-01','2024-03-04','2024-03-01',
                                  '2024-03-03','2024-03-01','2024-03-09','2024-05-01']),
    'checkout':   pd.to_datetime(['2024-03-05','2024-03-08','2024-03-03',
                                  '2024-03-07','2024-03-10','2024-03-12','2024-05-05'])
})
print(bookings)


**Concepts to use:**
1. Self-join on `room_id` → all booking pairs per room.
2. Filter `booking_id_a < booking_id_b` — avoids self-pairs and double-counting.
3. Overlap condition: `checkin_a < checkout_b AND checkin_b < checkout_a` (strict inequality).
4. `groupby('room_id')['overlap'].any()` → True if any double-booking.

In [ ]:
# Optimised Solution
def detect_double_bookings(df):
    merged = df.merge(df, on='room_id', suffixes=('_a','_b'))
    pairs = merged[merged['booking_id_a'] < merged['booking_id_b']].copy()
    pairs['overlap'] = (
        (pairs['checkin_a'] < pairs['checkout_b']) &
        (pairs['checkin_b'] < pairs['checkout_a'])
    )
    result = pairs.groupby('room_id')['overlap'].any().reset_index()
    result.columns = ['room_id','has_double_booking']
    # Add rooms with only 1 booking (no pairs → no double booking)
    all_rooms = df[['room_id']].drop_duplicates()
    return all_rooms.merge(result, on='room_id', how='left').fillna(False)

print(detect_double_bookings(bookings))


---
## Q3 — Average Ticket Resolution Time per Support Team (Hard)

**Concept:** filter on event_type · merge start/end · compute duration · groupby mean

**Problem:** The `ticket_events` table stores `opened` and `closed` events per ticket. Compute the **average resolution time in hours** per support team.

**Sample Input:**

| event_id | ticket_id | team    | event_type | timestamp           |
|----------|-----------|---------|------------|---------------------|
| 1        | TK1       | Alpha   | opened     | 2024-01-01 09:00:00 |
| 2        | TK1       | Alpha   | closed     | 2024-01-01 11:30:00 |
| 3        | TK2       | Alpha   | opened     | 2024-01-02 10:00:00 |
| 4        | TK2       | Alpha   | closed     | 2024-01-02 14:00:00 |
| 5        | TK3       | Beta    | opened     | 2024-01-01 08:00:00 |
| 6        | TK3       | Beta    | closed     | 2024-01-01 09:00:00 |

**Sample Output:**

| team  | avg_resolution_hrs |
|-------|--------------------|
| Alpha | 3.0                |
| Beta  | 1.0                |

> Alpha avg = (2.5h + 4h) / 2 = 3.25h · Beta = 1h

In [ ]:
import pandas as pd

ticket_events = pd.DataFrame({
    'event_id':   [1,2,3,4,5,6,7,8],
    'ticket_id':  ['TK1','TK1','TK2','TK2','TK3','TK3','TK4','TK4'],
    'team':       ['Alpha','Alpha','Alpha','Alpha','Beta','Beta','Beta','Beta'],
    'event_type': ['opened','closed','opened','closed','opened','closed','opened','closed'],
    'timestamp':  pd.to_datetime([
        '2024-01-01 09:00','2024-01-01 11:30',  # TK1: 2.5h
        '2024-01-02 10:00','2024-01-02 14:00',  # TK2: 4.0h
        '2024-01-01 08:00','2024-01-01 09:00',  # TK3: 1.0h
        '2024-01-03 07:00','2024-01-03 10:00',  # TK4: 3.0h
    ])
})
print(ticket_events)


**Concepts to use:**
1. Split into `opened` and `closed` DataFrames.
2. Merge on `['ticket_id','team']` with suffixes `_open` / `_close`.
3. `(ts_close - ts_open).dt.total_seconds() / 3600` → hours.
4. `groupby('team')['resolution_hrs'].mean().round(2)`.

In [ ]:
# Optimised Solution
def avg_resolution_time(df):
    opened = df[df['event_type']=='opened'][['ticket_id','team','timestamp']]
    closed = df[df['event_type']=='closed'][['ticket_id','team','timestamp']]
    merged = opened.merge(closed, on=['ticket_id','team'], suffixes=('_open','_close'))
    merged['resolution_hrs'] = (merged['timestamp_close'] - merged['timestamp_open']).dt.total_seconds() / 3600
    return merged.groupby('team')['resolution_hrs'].mean().round(2).reset_index()

print(avg_resolution_time(ticket_events))


---
## Q4 — Merge Monthly Sales Reports and Compute Annual Summary (Medium/Advanced)

**Concept:** glob · pd.concat · groupby agg post-merge

**Problem:** You receive one CSV per month named `sales_YYYY_MM.csv`. Merge all months into one DataFrame and compute total annual revenue and total units sold per product.

**Sample Input (3 mock files):**

sales_2024_01.csv:

| product | revenue | units |
|---------|---------|-------|
| A       | 1000    | 10    |
| B       | 500     | 5     |

sales_2024_02.csv / sales_2024_03.csv: similar structure

**Sample Output:**

| product | total_revenue | total_units |
|---------|---------------|-------------|
| A       | 3000          | 30          |
| B       | 1500          | 15          |

In [ ]:
import pandas as pd
import os, glob

# Setup: create mock monthly sales CSVs
os.makedirs('/tmp/monthly_sales', exist_ok=True)
for month, mult in [('01', 1), ('02', 1.2), ('03', 0.9)]:
    df = pd.DataFrame({
        'product': ['A', 'B', 'C'],
        'revenue': [int(1000*mult), int(500*mult), int(300*mult)],
        'units':   [int(10*mult),  int(5*mult),   int(3*mult)]
    })
    df.to_csv(f'/tmp/monthly_sales/sales_2024_{month}.csv', index=False)
print('Created mock monthly CSVs')


**Concepts to use:**
1. `glob.glob(path + '/*.csv')` — discover all monthly files.
2. `pd.concat([pd.read_csv(f) for f in files], ignore_index=True)` — merge.
3. `groupby('product').agg(total_revenue=('revenue','sum'), total_units=('units','sum'))`.

In [ ]:
# Optimised Solution
def annual_summary(directory):
    files = glob.glob(os.path.join(directory, '*.csv'))
    combined = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    return combined.groupby('product').agg(
        total_revenue=('revenue','sum'),
        total_units=('units','sum')
    ).reset_index()

print(annual_summary('/tmp/monthly_sales'))


---
## Q5 — Driver Trip Stats with Multi-Agg Lambda (Medium/Advanced)

**Concept:** named aggregation · lambda within agg · multiple metrics in one pass

**Problem:** Compute per driver: total trips, average trip duration (mins), count of long trips (>30 mins), and maximum trip distance (km).

**Sample Input:**

| trip_id | driver_id | duration_mins | distance_km |
|---------|-----------|---------------|-------------|
| T1      | D1        | 25            | 10          |
| T2      | D1        | 45            | 20          |
| T3      | D1        | 35            | 15          |
| T4      | D2        | 15            | 5           |
| T5      | D2        | 60            | 30          |

**Sample Output:**

| driver_id | total_trips | avg_duration | long_trips | max_distance |
|-----------|-------------|--------------|------------|--------------|
| D1        | 3           | 35.0         | 2          | 20           |
| D2        | 2           | 37.5         | 1          | 30           |

In [ ]:
import pandas as pd

trips = pd.DataFrame({
    'trip_id':      ['T1','T2','T3','T4','T5','T6','T7'],
    'driver_id':    ['D1','D1','D1','D2','D2','D3','D3'],
    'duration_mins':[25,  45,  35,  15,  60,  20,  55],
    'distance_km':  [10,  20,  15,  5,   30,  8,   25]
})
print(trips)


**Concepts to use:**
1. `groupby('driver_id').agg(name=('col', func))` — named aggregation syntax.
2. `long_trips=('duration_mins', lambda x: (x > 30).sum())` — lambda inside agg.
3. All 4 metrics computed in one groupby pass — efficient.

In [ ]:
# Optimised Solution
def driver_stats(df):
    return df.groupby('driver_id').agg(
        total_trips   = ('trip_id',      'count'),
        avg_duration  = ('duration_mins', 'mean'),
        long_trips    = ('duration_mins', lambda x: (x > 30).sum()),
        max_distance  = ('distance_km',   'max')
    ).reset_index()

print(driver_stats(trips))


---
## Q6 — Parse Server Logs to Find Peak Hour per Endpoint (Medium)

**Concept:** str.strip/split · pd.to_datetime · dt.hour · groupby + idxmax

**Problem:** Raw server logs are messy strings. Extract the `endpoint` and `timestamp`, then find the **peak hour (most requests) for each endpoint**.

**Sample Input:**

| log_id | raw_log                                         |
|--------|-------------------------------------------------|
| 1      | ' GET /api/users 2024-01-15 08:30:00'           |
| 2      | 'POST /api/orders  2024-01-15 08:45:00 '        |
| 3      | ' GET /api/users 2024-01-15 09:10:00'           |
| 4      | ' GET /api/users 2024-01-15 08:55:00'           |

**Sample Output:**

| endpoint    | peak_hour | request_count |
|-------------|-----------|---------------|
| /api/orders | 8         | 1             |
| /api/users  | 8         | 2             |

> /api/users has 2 requests in hour 8 vs 1 in hour 9 → peak = 8

In [ ]:
import pandas as pd

logs = pd.DataFrame({
    'log_id': [1,2,3,4,5,6,7,8],
    'raw_log': [
        ' GET /api/users 2024-01-15 08:30:00',
        'POST /api/orders  2024-01-15 08:45:00 ',
        ' GET /api/users 2024-01-15 09:10:00',
        ' GET /api/users 2024-01-15 08:55:00',
        'DELETE /api/users 2024-01-15 09:20:00',
        'POST /api/orders 2024-01-15 10:00:00',
        ' GET /api/users 2024-01-15 09:45:00',
        'POST /api/orders 2024-01-15 10:30:00',
    ]
})
print(logs)


**Concepts to use:**
1. `str.strip().str.split()` — clean and tokenise each log line.
2. `apply(lambda p: p[1])` → endpoint; `' '.join(p[2:4])` → timestamp string.
3. `pd.to_datetime(timestamp_str)` → `.dt.hour`.
4. `groupby(['endpoint','hour']).size()` → `idxmax()` per endpoint.

In [ ]:
# Optimised Solution
def peak_hour_per_endpoint(df):
    df = df.copy()
    parsed = df['raw_log'].str.strip().str.split()
    df['endpoint']  = parsed.apply(lambda p: p[1])
    df['timestamp'] = pd.to_datetime(parsed.apply(lambda p: ' '.join(p[2:4])))
    df['hour'] = df['timestamp'].dt.hour
    counts = df.groupby(['endpoint','hour']).size().reset_index(name='request_count')
    idx = counts.groupby('endpoint')['request_count'].idxmax()
    return counts.loc[idx].reset_index(drop=True)

print(peak_hour_per_endpoint(logs))


---
## Q7 — Sort Tasks by Priority, Deadline, then Name (Medium)

**Concept:** sorted with multi-key lambda · negative for descending · stable sort

**Problem:** Sort a task list by: 1) **priority descending** (High > Medium > Low), 2) **deadline ascending** (earliest first), 3) **task name alphabetically** for remaining ties.

**Sample Input:**

| task_id | name       | priority | deadline   |
|---------|------------|----------|------------|
| T1      | Auth fix   | High     | 2024-02-01 |
| T2      | UI update  | Medium   | 2024-01-15 |
| T3      | DB migrate | High     | 2024-01-20 |
| T4      | Logging    | Low      | 2024-01-10 |
| T5      | API docs   | Medium   | 2024-01-15 |

**Sample Output (sorted):**

| task_id | name       | priority | deadline   |
|---------|------------|----------|------------|
| T3      | DB migrate | High     | 2024-01-20 |
| T1      | Auth fix   | High     | 2024-02-01 |
| T5      | API docs   | Medium   | 2024-01-15 |
| T2      | UI update  | Medium   | 2024-01-15 |
| T4      | Logging    | Low      | 2024-01-10 |

> High first · within High: earlier deadline first · Medium tie on date: API docs < UI update alphabetically

In [ ]:
import pandas as pd

tasks = pd.DataFrame({
    'task_id':  ['T1','T2','T3','T4','T5','T6'],
    'name':     ['Auth fix','UI update','DB migrate','Logging','API docs','Cache clear'],
    'priority': ['High','Medium','High','Low','Medium','Low'],
    'deadline': pd.to_datetime(['2024-02-01','2024-01-15','2024-01-20','2024-01-10','2024-01-15','2024-01-10'])
})
print(tasks)


**Concepts to use:**
1. Map priority to numeric: `{'High':3,'Medium':2,'Low':1}` — enables numeric sorting.
2. `sort_values(['priority_num','deadline','name'], ascending=[False, True, True])`.
3. Alternative: `sorted()` with `key=lambda r: (-priority_map[r.priority], r.deadline, r.name)`.

In [ ]:
# Optimised Solution
def sort_tasks(df):
    df = df.copy()
    priority_map = {'High': 3, 'Medium': 2, 'Low': 1}
    df['priority_num'] = df['priority'].map(priority_map)
    return df.sort_values(
        ['priority_num','deadline','name'],
        ascending=[False, True, True]
    ).drop(columns='priority_num').reset_index(drop=True)

print(sort_tasks(tasks))


---
## Q8 — Top-K Most Commented Posts (Python heapq) (Medium/Advanced)

**Concept:** heapq.nlargest · key function · O(N log K) vs O(N log N)

**Problem:** From a list of posts, return the **top-K posts by comment count**. If two posts have the same comment count, rank by **likes descending** as a tiebreak.

**Sample Input:**

| post_id | title          | comments | likes |
|---------|----------------|----------|-------|
| P1      | Python Tips    | 120      | 500   |
| P2      | ML Basics      | 85       | 300   |
| P3      | SQL Tricks     | 120      | 650   |
| P4      | Data Viz Guide | 200      | 100   |
| P5      | Docker Intro   | 60       | 200   |

K = 3

**Sample Output:**

| rank | post_id | title          | comments | likes |
|------|---------|----------------|----------|-------|
| 1    | P4      | Data Viz Guide | 200      | 100   |
| 2    | P3      | SQL Tricks     | 120      | 650   |
| 3    | P1      | Python Tips    | 120      | 500   |

> P3 beats P1 on tiebreak (650 likes > 500 likes).

In [ ]:
import heapq

posts = [
    {'post_id':'P1','title':'Python Tips',    'comments':120,'likes':500},
    {'post_id':'P2','title':'ML Basics',      'comments':85, 'likes':300},
    {'post_id':'P3','title':'SQL Tricks',     'comments':120,'likes':650},
    {'post_id':'P4','title':'Data Viz Guide', 'comments':200,'likes':100},
    {'post_id':'P5','title':'Docker Intro',   'comments':60, 'likes':200},
    {'post_id':'P6','title':'Pandas Deep Dive','comments':85,'likes':400},
]
K = 3
print('Total posts:', len(posts), '| K:', K)


**Concepts to use:**
1. `heapq.nlargest(k, iterable, key)` — O(N log K), optimal for K << N.
2. `key=lambda p: (p['comments'], p['likes'])` — tuple key: primary = comments, secondary = likes.
3. Both are ascending in the tuple, both will be maxed → no negation needed with `nlargest`.

In [ ]:
# Optimised Solution
def top_k_posts(posts, k):
    return heapq.nlargest(k, posts, key=lambda p: (p['comments'], p['likes']))

for rank, post in enumerate(top_k_posts(posts, K), 1):
    print(f"Rank {rank}: {post['post_id']} — {post['title']} ({post['comments']} comments, {post['likes']} likes)")


---
## Q9 — Daily Game Score Lag, Lead & Gain (Advanced)

**Concept:** groupby().shift(1/−1) · pct_change within group · per-player computation

**Problem:** Track daily scores for each player in a mobile game. Add columns for yesterday's score (lag), tomorrow's score (lead), and the day-over-day point gain/loss.

**Sample Input:**

| player | date       | score |
|--------|------------|-------|
| P1     | 2024-01-01 | 100   |
| P1     | 2024-01-02 | 130   |
| P1     | 2024-01-03 | 120   |
| P2     | 2024-01-01 | 200   |
| P2     | 2024-01-02 | 180   |

**Sample Output:**

| player | date       | score | prev_score | next_score | daily_gain |
|--------|------------|-------|------------|------------|------------|
| P1     | 2024-01-01 | 100   | NaN        | 130        | NaN        |
| P1     | 2024-01-02 | 130   | 100        | 120        | +30        |
| P1     | 2024-01-03 | 120   | 130        | NaN        | -10        |
| P2     | 2024-01-01 | 200   | NaN        | 180        | NaN        |
| P2     | 2024-01-02 | 180   | 200        | NaN        | -20        |

In [ ]:
import pandas as pd

scores = pd.DataFrame({
    'player': ['P1','P1','P1','P2','P2','P2','P3','P3'],
    'date':   pd.to_datetime(['2024-01-01','2024-01-02','2024-01-03',
                              '2024-01-01','2024-01-02','2024-01-03',
                              '2024-01-01','2024-01-03']),  # P3 has a gap
    'score':  [100, 130, 120, 200, 180, 190, 300, 280]
})
print(scores)


**Concepts to use:**
1. Sort by `['player','date']` before shifting.
2. `groupby('player')['score'].shift(1)` → prev_score (lag-1).
3. `groupby('player')['score'].shift(-1)` → next_score (lead-1).
4. `score - prev_score` → daily_gain.

In [ ]:
# Optimised Solution
def score_lag_lead(df):
    df = df.sort_values(['player','date']).copy()
    grp = df.groupby('player')['score']
    df['prev_score'] = grp.shift(1)
    df['next_score'] = grp.shift(-1)
    df['daily_gain'] = df['score'] - df['prev_score']
    return df

print(score_lag_lead(scores))


---
## Q10 — Rank Students Within Each Class (rank vs dense_rank) (Advanced)

**Concept:** groupby().rank(method='min') vs rank(method='dense') · ascending=False · handle ties

**Problem:** Rank students by exam score within their class. Show both standard rank (with gaps for ties) and dense rank (no gaps). Top scorer in each class gets rank 1.

**Sample Input:**

| student_id | class | score |
|------------|-------|-------|
| S1         | A     | 95    |
| S2         | A     | 85    |
| S3         | A     | 85    |
| S4         | A     | 70    |
| S5         | B     | 90    |
| S6         | B     | 90    |
| S7         | B     | 75    |

**Sample Output:**

| student_id | class | score | rank_standard | rank_dense |
|------------|-------|-------|---------------|------------|
| S1         | A     | 95    | 1             | 1          |
| S2         | A     | 85    | 2             | 2          |
| S3         | A     | 85    | 2             | 2          |
| S4         | A     | 70    | 4             | 3          |
| S5         | B     | 90    | 1             | 1          |
| S6         | B     | 90    | 1             | 1          |
| S7         | B     | 75    | 3             | 2          |

> Standard rank: tie at 2 skips to 4 · Dense rank: tie at 1 goes to 2 (no gap)

In [ ]:
import pandas as pd

students = pd.DataFrame({
    'student_id': ['S1','S2','S3','S4','S5','S6','S7','S8'],
    'class':      ['A','A','A','A','B','B','B','B'],
    'score':      [95, 85, 85, 70, 90, 90, 75, 75]
})
print(students)


**Concepts to use:**
1. `groupby('class')['score'].rank(method='min', ascending=False)` → standard rank (gaps).
2. `rank(method='dense', ascending=False)` → dense rank (no gaps).
3. Use `ascending=False` because highest score = rank 1.

In [ ]:
# Optimised Solution
def rank_students(df):
    df = df.copy()
    df['rank_standard'] = df.groupby('class')['score'].rank(method='min',   ascending=False).astype(int)
    df['rank_dense']    = df.groupby('class')['score'].rank(method='dense',  ascending=False).astype(int)
    return df.sort_values(['class','rank_dense']).reset_index(drop=True)

print(rank_students(students))
